# MLPerf LLM Benchmark
Benchmarks LLM inference throughput and latency using **vLLM** (pre-installed in this image).
Models are loaded from the Hugging Face Hub — weights are cached under `~/.cache/huggingface`
so subsequent runs skip the download.

**Run cells top to bottom. Edit only Cell 2.**

## Cell 1 — GPU Verification

In [ ]:
import torch
print('=== GPU Status ===')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i} : {p.name}  {p.total_memory/1e9:.1f} GB')
    _ = torch.rand(512,512,device='cuda') @ torch.rand(512,512,device='cuda')
    torch.cuda.synchronize()
    print('Smoke test : PASSED')
else:
    print('ERROR: LLM benchmark requires a GPU. Stopping.')

## Cell 2 — Configuration
Edit this cell to select the model and run settings.

In [ ]:
# ── Edit these ──────────────────────────────────────────────────────────────
# Any Hugging Face model ID works. For gated models (e.g. Meta-Llama) you
# need to set HF_TOKEN in your environment before starting this container.
MODEL_HF_ID     = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # fast default
# MODEL_HF_ID   = 'meta-llama/Llama-3.1-8B-Instruct'    # gated — needs HF_TOKEN
# MODEL_HF_ID   = 'Qwen/Qwen2.5-7B-Instruct'            # ungated
# MODEL_HF_ID   = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'

TENSOR_PARALLEL = 1         # number of GPUs to use (tensor parallelism)
DTYPE           = 'auto'    # 'auto' | 'float16' | 'bfloat16'
MAX_MODEL_LEN   = None      # set an int (e.g. 4096) to cap context length

SCENARIO        = 'Offline' # 'Offline' | 'SingleStream'
NUM_WARMUP      = 5         # warmup requests (not measured)
NUM_BENCH       = 100       # benchmark requests
MAX_TOKENS      = 256       # max output tokens per request

LOG_DIR = '/tmp/mlperf_llm_results'
# ────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Model     : {MODEL_HF_ID}')
print(f'Scenario  : {SCENARIO}')
print(f'Requests  : {NUM_BENCH}  (+ {NUM_WARMUP} warmup)')
print(f'Max tokens: {MAX_TOKENS}')
print(f'TP        : {TENSOR_PARALLEL}')
print(f'Log dir   : {LOG_DIR}')

## Cell 3 — Verify Model Access
Checks whether the model is already in the HuggingFace cache. Downloads it if not.
Gated models (Meta Llama, etc.) require `HF_TOKEN` to be set in the environment.

In [ ]:
from huggingface_hub import snapshot_download, HfApi
import os

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    print('HF_TOKEN : set (gated models accessible)')
else:
    print('HF_TOKEN : not set (public models only)')

print(f'Checking cache for {MODEL_HF_ID} ...')
try:
    model_path = snapshot_download(
        repo_id=MODEL_HF_ID,
        token=hf_token,
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
    )
    print(f'Model ready: {model_path}')
except Exception as e:
    print(f'ERROR: {e}')
    if 'gated' in str(e).lower() or '401' in str(e):
        print('Set HF_TOKEN to a token from an account with model access.')
    raise

## Cell 4 — Run LLM Benchmark
Uses vLLM's offline inference API (`LLM.generate`) — no separate server process needed.
Measures throughput (tokens/sec) and mean latency across `NUM_BENCH` requests.

In [ ]:
import time, json, statistics, os
from vllm import LLM, SamplingParams

# Diverse synthetic prompts covering common NLP tasks.
# Repeated / cycled to fill NUM_BENCH + NUM_WARMUP requests.
_PROMPT_POOL = [
    'Explain the difference between machine learning and deep learning in simple terms.',
    'Write a short Python function that reverses a linked list.',
    'Summarize the key ideas behind the transformer architecture.',
    'What are the main causes of climate change?',
    'Describe three practical applications of large language models in healthcare.',
    'What is gradient descent and how does it work?',
    'Write a haiku about a GPU running inference at midnight.',
    'Explain the CAP theorem and when you would choose AP over CP.',
    'What are the trade-offs between RLHF and DPO for aligning LLMs?',
    'Describe how attention mechanisms work in transformer models.',
    'What is quantization and why does it matter for LLM deployment?',
    'Write a brief explanation of vector databases for a software engineer.',
]

total_needed = NUM_WARMUP + NUM_BENCH
prompts_all  = [_PROMPT_POOL[i % len(_PROMPT_POOL)] for i in range(total_needed)]

print(f'Loading model: {MODEL_HF_ID}')
llm_kwargs = dict(
    model=MODEL_HF_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
)
if MAX_MODEL_LEN:
    llm_kwargs['max_model_len'] = MAX_MODEL_LEN

llm    = LLM(**llm_kwargs)
params = SamplingParams(temperature=0.6, top_p=0.9, max_tokens=MAX_TOKENS)

# ── Warmup ──────────────────────────────────────────────────────────────────
print(f'Warming up ({NUM_WARMUP} requests) ...')
_ = llm.generate(prompts_all[:NUM_WARMUP], params)
torch_dev = 0
try:
    import torch; torch.cuda.synchronize()
except Exception:
    pass

# ── Benchmark ───────────────────────────────────────────────────────────────
print(f'Benchmarking ({NUM_BENCH} requests) ...')
bench_prompts = prompts_all[NUM_WARMUP:]

t_start  = time.perf_counter()
outputs  = llm.generate(bench_prompts, params)
t_end    = time.perf_counter()

elapsed  = t_end - t_start

# ── Metrics ─────────────────────────────────────────────────────────────────
out_lens = [len(o.outputs[0].token_ids) for o in outputs]
total_out_tokens = sum(out_lens)

# Approximate input tokens (prompt chars / 4 is a rough heuristic)
in_lens  = [len(p) // 4 for p in bench_prompts]
total_in_tokens  = sum(in_lens)

output_throughput = total_out_tokens / elapsed
total_throughput  = (total_in_tokens + total_out_tokens) / elapsed
mean_latency_ms   = (elapsed / NUM_BENCH) * 1000

results = {
    'model':              MODEL_HF_ID,
    'scenario':           SCENARIO,
    'num_requests':       NUM_BENCH,
    'elapsed_s':          round(elapsed, 3),
    'total_input_tokens': total_in_tokens,
    'total_output_tokens':total_out_tokens,
    'output_throughput_tok_s': round(output_throughput, 2),
    'total_throughput_tok_s':  round(total_throughput, 2),
    'mean_latency_ms':    round(mean_latency_ms, 2),
    'min_out_tokens':     min(out_lens),
    'max_out_tokens':     max(out_lens),
    'mean_out_tokens':    round(statistics.mean(out_lens), 1),
}

# Save to log dir
out_path = os.path.join(LOG_DIR, 'results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved: {out_path}')

## Cell 5 — Results

In [ ]:
import json, os

out_path = os.path.join(LOG_DIR, 'results.json')
if not os.path.exists(out_path):
    print('No results file found — check Cell 4 output.')
else:
    r = json.load(open(out_path))
    print('=== LLM Benchmark Results ===')
    print(f"  Model             : {r['model']}")
    print(f"  Scenario          : {r['scenario']}")
    print(f"  Requests          : {r['num_requests']}")
    print(f"  Elapsed           : {r['elapsed_s']} s")
    print(f"  Output throughput : {r['output_throughput_tok_s']} tok/s")
    print(f"  Total throughput  : {r['total_throughput_tok_s']} tok/s  (in+out)")
    print(f"  Mean latency      : {r['mean_latency_ms']} ms / request")
    print(f"  Output tokens     : min={r['min_out_tokens']}  max={r['max_out_tokens']}  mean={r['mean_out_tokens']}")
    print()
    print('Full results:')
    print(json.dumps(r, indent=2))